MLflow es una plataforma open-source diseñada para gestionar el end-to-end lifecycle de machine learning. Desarrollada por Databricks, viene pre-installed en el Databricks Runtime for ML. MLflow consiste en cuatro componentes principales: model tracking, model packaging, y model registry.
1. Model Tracking: Registra y consulta experimentos, incluyendo código, datos, configuración y resultados.
2. Models: Ofrece un formato de modelo general compatible con diversas herramientas de deployment.
3. Model Registry: Una solución centralizada y colaborativa para gestionar todo el model lifecycle. 
4. MLflow Projects: Un formato para empaquetar código de ciencia de datos de manera reusable y reproducible.

Conceptos relacionados a model tracking
* Parameters: hiperparámetros, cantidad arbitraria de parámetros.
* Metrics: métricas de evaluación... en regresión, registra tu r cuadrada.
* Artifacts: archivos arbitrarios... lo que quieras... archivos CSV de feature importance, archivos de imagen, etc.
* Source: notebook que ejecutó el código, proyectos/código en GitHub.

In [0]:
from sklearn.ensemble import IsolationForest
import mlflow 
from mlflow.models.signature import infer_signature

In [0]:
%sql
create schema if not exists dev.models;

In [0]:
mlflow.set_registry_uri("databricks-uc")
experiment_name = "Anomaly Detection Exp"
user_email = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
full_path = f"/Users/{user_email}/{experiment_name}"
exp_id=mlflow.create_experiment( name=full_path)
mlflow.set_experiment(experiment_id=exp_id)
#1387806134246030

In [0]:
df_sessions=mlflow.data.load_delta(table_name="dev.feature_store.sessions_arkime",name="sessions_arkime")

In [0]:
df=df_sessions.df.select(['dst_data_bytes',
 'tot_data_bytes',
 'packetlen_min',
 'packetlen_max',
 'packetlen_media',
 'session_duration'])
df_pandas=df.toPandas()
df_pandas.head()

In [0]:
client=mlflow.MlflowClient()
def get_latest_model_version(model):
    model_version_infos = client.search_model_versions("name='%s'"%model_name)
    return max([model_version_info.version for model_version_info in model_version_infos])

Un MLflow Experiment es una unidad organizativa de alto nivel. Piensa en él como un contenedor que alberga un conjunto de runs. Es donde agrupas y organizas runs relacionados, generalmente realizados para explorar diferentes configuraciones, parámetros o algoritmos.
Tienes dos opciones:
* Notebook Experiment: Está vinculado al notebook en el que estás trabajando, proporcionando un enfoque más localizado.
* Workspace Experiment: Te permite especificar manualmente el path, ofreciendo mayor flexibilidad. Múltiples notebooks pueden aportar runs a un workspace experiment, convirtiéndolo en un hub colaborativo.

In [0]:
model_name = "dev.models.isolation_forest"
with mlflow.start_run(run_name="Isolation Forest") as run:
    params={"n_estimators":100,"max_samples":"auto","random_state":42}
    mlflow.log_input(df_sessions,context="source")
    mlflow.log_input(mlflow.data.from_pandas(df_pandas,source=df_sessions.source),context="train")
    mlflow.log_params(params)
    mod_isolation=IsolationForest(**params).fit(df_pandas)
    y_pred=mod_isolation.predict(df_pandas)
    sign=infer_signature(df_pandas,y_pred)
    mlflow.sklearn.log_model(
        sk_model=mod_isolation,
        artifact_path="isolation_model",
        registered_model_name=model_name,
        signature=sign
    )
client.set_registered_model_alias(model_name,"champion",get_latest_model_version(model_name))

In [0]:
run.info